# 🛠️ AX 실습 PC 환경 셋팅 (Jupyter 노트북)

이 Jupyter 노트북은 오늘 실습에 필요한 PC 환경을 **위에서부터 차례로** 점검하고, 빠진 것을 채웁니다.

**사용법**
1. 회색 칸(코드 셀) 왼쪽의 **▶ 버튼**을 누르면 그 단계가 실행됩니다. (단축키: `Shift + Enter`)
2. 처음 실행하면 오른쪽 위에 **커널 선택**(노트북을 실행할 Python 고르기) 창이 뜹니다 → 사전 안내 시간에 설치한 Python을 고르세요.
3. 결과는 세 가지로 표시됩니다.
   - ✅ 정상 → 다음 단계로
   - ❌ 문제 있음 → 👉 안내대로 한 뒤 **같은 셀을 다시 실행**
   - ⚠️ 아직 정해지지 않은 값이 있음 → 강사 안내를 기다리기
4. 어떤 셀이든 **여러 번 실행해도 괜찮습니다.**
5. 같은 ❌가 두 번 나오면 손을 들어 보조강사를 불러 주세요.

> 🔒 **Virtual Key(사내 Gateway 이용 열쇠)는 이 노트북에 절대 적지 마세요.** 10단계에서 `.env` 파일에만 넣습니다.

> 📋 맨 아래에는 실습 시간에 복사해서 쓰는 **실습 프롬프트 모음**이 있습니다.

### 0단계. 준비: 공통 설정 불러오기
**무엇을 하나요?** 이후 단계에서 함께 쓰는 설정값과 도우미 기능을 불러옵니다. 설치나 변경은 하지 않습니다.

**정상이면** `✅ 준비 완료 (Python 3.x.x)`

**실패하면** 커널 선택 창에서 Python을 골랐는지 확인하고 다시 실행합니다. 목록에 Python이 없으면 VS Code의 **Python·Jupyter 확장**이 설치됐는지 확인합니다(사전 안내 시간에 설치).

> 강사용: 아래 `[확인 필요: …]` 값은 교육 전에 강사가 채워서 배포합니다.

In [ ]:
# 0단계: 준비 (공통 설정과 도우미 기능)
import os, sys, json, shutil, subprocess, platform
from pathlib import Path

# ── 강사가 채울 값 ─────────────────────────────────────────
PRACTICE_DIR = Path.home() / "ax-training"               # [확인 필요: 기본 폴더 위치]
SKILL_NAME = "[확인 필요: Skill 이름]"
SKILL_SOURCE = r"[확인 필요: Skill 배포 위치]"           # 배포받은 Skill 폴더 경로
SKILL_INSTALL_DIR = Path.home() / ".claude" / "skills"  # [확인 필요: Skill 설치 위치]
REQUIRED_PACKAGES = []                                   # [확인 필요: Skill에 필요한 Python 패키지] 예: ["requests"]
NEED_NODE = None                                         # [확인 필요: Node.js 필요 여부] True 또는 False
ENV_URL_NAME = "[확인 필요: Gateway 주소 변수명]"
ENV_KEY_NAME = "[확인 필요: Virtual Key 변수명]"
# ─────────────────────────────────────────────────────────

STEP_ORDER = ["준비", "Python", "VS Code 확장", "Windows 11", "VS Code", "Claude Code", "Bedrock 설정", "Python 패키지",
              "Node.js", "실습 폴더", "Skill", ".env", "Gateway 연결", "Splunk 조회", "일정 조회"]
MARK = {True: "✅", False: "❌", None: "⚠️"}
RESULTS = globals().get("RESULTS", {})

def show(step, ok, msg, hint=""):
    # 결과를 ✅/❌/⚠️ 로 보여주고 마지막 요약용으로 기록한다
    print(f"{MARK[ok]} {msg}")
    if hint and ok is not True:
        print(f"   👉 {hint}")
    RESULTS[step] = (ok, msg)

def run(cmd, timeout=60):
    # 명령을 실행하고 (성공 여부, 출력 내용)을 돌려준다
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, encoding="utf-8", errors="replace")
        return p.returncode == 0, ((p.stdout or "") + (p.stderr or "")).strip()
    except FileNotFoundError:
        return False, "명령을 찾을 수 없음"
    except subprocess.TimeoutExpired:
        return False, "시간 초과"

def is_placeholder(value):
    return isinstance(value, str) and value.startswith("[확인 필요")

def read_env(path):
    # .env 파일을 읽어 {변수명: 값} 으로 돌려준다 (값은 화면에 출력하지 않는다)
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8-sig", errors="replace").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                values[k.strip()] = v.strip().strip('"').strip("'")
    return values

show("준비", True, f"준비 완료 (Python {platform.python_version()})")

### 1단계. Python·VS Code 확장 확인
**무엇을 하나요?** 사전 안내 시간에 설치한 Python과 VS Code 확장(Python, Jupyter)의 버전을 확인합니다. 이 셀이 실행됐다면 Jupyter 노트북 실행 환경도 정상입니다.

**정상이면**
```
✅ Python 3.x.x — 위치: C:\Users\...\python.exe
✅ VS Code 확장 확인: Python 20xx.x.x, Jupyter 20xx.x.x
```

**실패하면**
- 셀이 아예 실행되지 않으면 Python이 없는 것입니다 → 배포받은 Python 설치파일을 실행합니다(첫 화면의 **Add python.exe to PATH** 체크). 그다음 VS Code를 완전히 닫았다 다시 열고 0단계부터 실행합니다.
- "ipykernel 패키지가 필요합니다" 창이 뜨면 **설치**를 누릅니다.
- 확장이 ❌이면 VS Code 왼쪽의 확장 아이콘(`Ctrl + Shift + X`)에서 빠진 확장(Microsoft의 Python 또는 Jupyter)을 설치하고 이 셀을 다시 실행합니다.

In [ ]:
# 1단계: Python(3.8 이상)과 VS Code 확장(Python, Jupyter) 확인
ok = sys.version_info >= (3, 8)
show("Python", ok, f"Python {platform.python_version()} — 위치: {sys.executable}",
     "Python 3.8 이상이 필요합니다. 배포받은 Python 설치파일로 다시 설치한 뒤 VS Code를 재시작하세요.")

NEED_EXTENSIONS = {"ms-python.python": "Python", "ms-toolsai.jupyter": "Jupyter"}

def installed_extensions():
    # VS Code 확장 목록 {확장 id: 버전}. code 명령으로 먼저 보고, 없으면 확장 폴더의 목록 파일을 읽는다
    code_path = shutil.which("code")
    if code_path:
        ok, out = run([code_path, "--list-extensions", "--show-versions"])
        if ok:
            return dict(line.strip().lower().split("@", 1) for line in out.splitlines() if "@" in line)
    ext_file = Path.home() / ".vscode" / "extensions" / "extensions.json"
    try:
        return {e["identifier"]["id"].lower(): e.get("version", "?") for e in json.loads(ext_file.read_text(encoding="utf-8"))}
    except Exception:
        return None

installed = installed_extensions()
if installed is None:
    show("VS Code 확장", False, "VS Code 확장 목록을 확인하지 못했습니다.",
         "VS Code를 완전히 닫았다가 다시 열고 이 셀을 다시 실행하세요. 그래도 ❌이면 보조강사를 불러 주세요.")
else:
    found = [f"{name} {installed[ext_id]}" for ext_id, name in NEED_EXTENSIONS.items() if ext_id in installed]
    missing = [name for ext_id, name in NEED_EXTENSIONS.items() if ext_id not in installed]
    show("VS Code 확장", not missing,
         f"VS Code 확장 확인: {', '.join(found)}" if not missing else f"VS Code 확장이 빠졌습니다: {', '.join(missing)}",
         "VS Code 왼쪽 확장 아이콘(Ctrl + Shift + X)에서 Microsoft의 확장을 설치하고 이 셀을 다시 실행하세요.")

### 2단계. Windows 11 확인
**무엇을 하나요?** 사전 배포 기준인 Windows 11이 맞는지 확인합니다. 확인만 하고 바꾸는 것은 없습니다.

**정상이면** `✅ Windows 11 (빌드 22xxx 이상)`

**실패하면** 실습 자체는 진행할 수 있는 경우가 많습니다. 보조강사에게 알려 주고 다음 단계로 넘어가세요.

In [ ]:
# 2단계: Windows 11 확인 (빌드 번호 22000 이상이면 Windows 11)
build = sys.getwindowsversion().build if sys.platform == "win32" else 0
show("Windows 11", build >= 22000,
     f"Windows 11 (빌드 {build})" if build >= 22000 else f"Windows 11이 아닙니다 (빌드 {build or '확인 불가'})",
     "사전 배포 기준과 다릅니다. 보조강사에게 알리고 다음 단계로 넘어가세요.")

### 3단계. VS Code 확인
**무엇을 하나요?** 사전 배포된 VS Code의 버전을 확인합니다.

**정상이면** `✅ VS Code 1.xx.x`

**실패하면** VS Code를 완전히 닫았다 다시 열고 이 셀을 다시 실행합니다. 그래도 ❌이면 보조강사를 부릅니다.

In [ ]:
# 3단계: VS Code 확인
path = shutil.which("code")
if path:
    ok, out = run([path, "--version"])
    show("VS Code", ok, f"VS Code {out.splitlines()[0] if ok and out else '버전 확인 실패'}",
         "VS Code를 완전히 닫았다가 다시 열고 이 셀을 다시 실행하세요.")
else:
    show("VS Code", False, "VS Code 명령(code)을 찾지 못했습니다.",
         "VS Code를 완전히 닫았다가 다시 열고 다시 실행하세요. 그래도 ❌이면 보조강사를 불러 주세요.")

### 4단계. Claude Code 확인
**무엇을 하나요?** 사전 배포된 Claude Code(터미널에서 대화로 일을 시키는 AI 도우미)가 설치돼 있는지, 버전이 무엇인지 확인합니다.

**정상이면** `✅ Claude Code 2.x.x (Claude Code)`

**실패하면** 사전 배포가 빠진 경우입니다. 보조강사를 불러 주세요.

In [ ]:
# 4단계: Claude Code 확인
path = shutil.which("claude")
fallback = Path.home() / ".local" / "bin" / "claude.exe"
if path:
    ok, out = run([path, "--version"])
    show("Claude Code", ok, f"Claude Code {out.splitlines()[0] if ok and out else '버전 확인 실패'}",
         "보조강사를 불러 주세요.")
elif fallback.exists():
    show("Claude Code", False, f"Claude Code는 있지만 PATH(명령을 찾는 경로)에 등록되지 않았습니다: {fallback}",
         "보조강사를 불러 주세요. [확인 필요: Claude Code 설치 방식]")
else:
    show("Claude Code", False, "Claude Code를 찾지 못했습니다(사전 배포 누락).",
         "보조강사를 불러 주세요. [확인 필요: Claude Code 설치 방법]")

### 5단계. Bedrock 연결 설정 확인
**무엇을 하나요?** Claude Code가 AWS Bedrock(아마존의 AI 모델 서비스)을 쓰도록 설정됐는지 확인합니다. 확인만 하고 바꾸는 것은 없습니다.

**정상이면** `✅ Bedrock 사용 설정 확인 (환경변수 또는 settings.json)`

**⚠️가 나오면** 사내 설정 방식이 이 확인 방법과 다를 수 있습니다. 8단계에서 Claude Code가 대답하면 정상이므로 다음 단계로 넘어가세요.

In [ ]:
# 5단계: Bedrock 연결 설정 확인 (환경변수 → Claude Code 설정 파일 순서로 찾는다)
# [확인 필요: Bedrock 연결 확인 방법] 사내 표준 방식에 맞게 조정
keys = ("CLAUDE_CODE_USE_BEDROCK", "AWS_REGION")
found = {k: (os.environ[k], "환경변수") for k in keys if os.environ.get(k)}
settings_file = Path.home() / ".claude" / "settings.json"
try:
    settings_env = json.loads(settings_file.read_text(encoding="utf-8")).get("env", {}) if settings_file.exists() else {}
except Exception:
    settings_env = {}
for k in keys:
    if k not in found and settings_env.get(k):
        found[k] = (str(settings_env[k]), "settings.json")

use, where = found.get("CLAUDE_CODE_USE_BEDROCK", ("", ""))
if use.lower() in ("1", "true"):
    region = found.get("AWS_REGION", ("미확인", ""))[0]
    show("Bedrock 설정", True, f"Bedrock 사용 설정 확인 ({where}), 리전: {region}")
else:
    show("Bedrock 설정", None, "이 방법으로는 Bedrock 설정을 찾지 못했습니다.",
         "사내 설정 방식이 다를 수 있습니다. 8단계에서 Claude Code가 대답하면 정상입니다.")

### 6단계. Python 패키지 확인·설치
**무엇을 하나요?** 실습용 Skill이 쓰는 Python 패키지(추가 기능 묶음)가 있는지 확인하고, 없으면 설치한 뒤 다시 확인합니다.

**정상이면** `✅ 모든 패키지 설치됨: …`

**실패하면** 출력된 오류를 보조강사에게 보여 주세요. 사내 프록시나 저장소 설정이 필요할 수 있습니다.

In [ ]:
# 6단계: Python 패키지 확인 → 없으면 설치 → 다시 확인
from importlib import metadata

def missing_packages():
    miss = []
    for name in REQUIRED_PACKAGES:
        try:
            metadata.version(name)
        except metadata.PackageNotFoundError:
            miss.append(name)
    return miss

if not REQUIRED_PACKAGES:
    show("Python 패키지", None, "설치할 패키지 목록이 아직 정해지지 않았습니다.", "강사 안내를 기다리세요.")
else:
    miss = missing_packages()
    if miss:
        print(f"설치 중: {', '.join(miss)} ...")
        ok, out = run([sys.executable, "-m", "pip", "install", *miss], timeout=600)
        if not ok:
            print(out[-1500:])
    miss = missing_packages()
    show("Python 패키지", not miss,
         f"모든 패키지 설치됨: {', '.join(REQUIRED_PACKAGES)}" if not miss else f"설치되지 않은 패키지: {', '.join(miss)}",
         "위 오류를 보조강사에게 보여 주세요. [확인 필요: 사내 pip 저장소/프록시]")

### 7단계. Node.js 확인·설치 (필요한 경우만)
**무엇을 하나요?** Node.js(일부 도구가 쓰는 실행 프로그램)가 이번 실습에 필요하면 확인하고, 없으면 설치한 뒤 다시 확인합니다. 필요 없으면 건너뜁니다.

**정상이면** `✅ Node.js v2x.x.x` 또는 `✅ Node.js는 이번 실습에 필요 없습니다`

**실패하면** 설치 직후라면 VS Code를 완전히 닫았다 다시 열고 0단계부터 다시 실행합니다. 그래도 ❌이면 보조강사를 부릅니다.

In [ ]:
# 7단계: Node.js 확인 → 없으면 설치(winget) → 다시 확인
def node_version():
    for p in (shutil.which("node"), r"C:\Program Files\nodejs\node.exe"):
        if p and Path(p).exists():
            ok, out = run([p, "--version"])
            if ok:
                return out
    return None

if NEED_NODE is None:
    show("Node.js", None, "Node.js가 필요한지 아직 정해지지 않았습니다.", "강사 안내를 기다리세요.")
elif NEED_NODE is False:
    show("Node.js", True, "Node.js는 이번 실습에 필요 없습니다(건너뜀).")
else:
    ver = node_version()
    if not ver:
        winget = shutil.which("winget")
        if winget:
            print("Node.js 설치 중 (몇 분 걸릴 수 있습니다) ...")
            ok, out = run([winget, "install", "--id", "OpenJS.NodeJS.LTS", "-e", "--silent",
                           "--accept-package-agreements", "--accept-source-agreements"], timeout=900)
            print(out[-1500:])
        else:
            print("winget(윈도우 설치 도구)을 찾지 못했습니다.")
        ver = node_version()
    show("Node.js", bool(ver), f"Node.js {ver}" if ver else "Node.js가 아직 없습니다.",
         "VS Code를 완전히 닫았다 다시 열고 0단계부터 다시 실행하세요. 그래도 ❌이면 보조강사를 불러 주세요.")

### 8단계. Claude Code 처음 실행하기 + 실습 폴더 만들기
**무엇을 하나요?** VS Code 터미널에서 실습 폴더 `ax-training`을 만들고, 그 안에서 Claude Code를 처음 실행합니다. 이 폴더가 오늘 Claude Code의 작업 공간입니다.

**따라 하기**
1. VS Code 위쪽 메뉴에서 **터미널 → 새 터미널**을 엽니다. (단축키: `` Ctrl + Shift + ` ``)
2. 아래를 한 줄씩 입력하고 Enter를 누릅니다. `mkdir`에서 "이미 있음" 오류가 나오면 무시하고 다음 줄로 넘어갑니다.

```powershell
cd ~
mkdir ax-training
cd ax-training
claude
```

3. 처음 실행하면 화면 색상(테마) 같은 설정을 묻습니다 → **Enter**로 기본값을 고릅니다.
4. "이 폴더를 믿을지" 묻는 화면이 나오면 → **Yes**
5. 입력창에 `안녕! 너는 무엇을 도와줄 수 있어?`를 입력합니다 → 대답이 나오면 AI 연결 성공
6. 이 노트북으로 돌아와 아래 셀을 실행합니다. 실습 폴더를 확인하고, Claude Code가 `.env`(10단계에서 만드는 키 파일)를 읽지 못하게 막는 설정(`.claude\settings.json`)을 만듭니다.

**정상이면** `✅ 실습 폴더 확인: C:\Users\...\ax-training (.env 읽기 금지 설정됨)`

이 설정은 9단계에서 Claude Code를 다시 켤 때 적용됩니다.

**실패하면** ❌이면 2번 명령을 다시 입력합니다. Claude Code가 대답하지 않거나 오류가 나면 화면을 그대로 두고 보조강사를 부릅니다.

> 기본 폴더 위치: [확인 필요: 기본 폴더 위치]

In [ ]:
# 8단계: 실습 폴더 확인 (폴더는 위 안내대로 터미널에서 직접 만든다)
#        + Claude Code가 .env(키 파일)를 읽지 못하게 실습 폴더 설정에 읽기 금지 규칙을 넣는다
ENV_DENY = ["Read(./.env)", "Read(./.env.*)"]

if not PRACTICE_DIR.is_dir():
    show("실습 폴더", False, f"실습 폴더가 없습니다: {PRACTICE_DIR}",
         "위 따라 하기 2번의 명령을 VS Code 터미널에 입력한 뒤 이 셀을 다시 실행하세요.")
else:
    settings_path = PRACTICE_DIR / ".claude" / "settings.json"
    try:
        settings = json.loads(settings_path.read_text(encoding="utf-8")) if settings_path.exists() else {}
        deny = settings.setdefault("permissions", {}).setdefault("deny", [])
        deny += [rule for rule in ENV_DENY if rule not in deny]
        settings_path.parent.mkdir(exist_ok=True)
        settings_path.write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
        show("실습 폴더", True, f"실습 폴더 확인: {PRACTICE_DIR} (.env 읽기 금지 설정됨)")
    except Exception as e:
        show("실습 폴더", False, f"실습 폴더는 있지만 .env 읽기 금지 설정을 만들지 못했습니다: {e}",
             "화면을 그대로 두고 보조강사를 불러 주세요.")

### 9단계. 실습용 Skill 설치
**무엇을 하나요?** Skill(Claude Code에 넣어 두는 업무 설명서)은 사내 Model/MCP Gateway를 쉽게 쓰게 해 줍니다. 설치돼 있는지 확인하고, 없으면 설치한 뒤 다시 확인합니다.

**정상이면** `✅ Skill 설치됨: …`

**설치 후 할 일** Claude Code 화면에서 `/exit`를 입력해 끄고, 다시 `claude`를 입력해 켭니다. 그다음 `사용할 수 있는 Skill 목록 알려줘`라고 물어 Skill 이름이 보이면 정상입니다.

**실패하면** 보조강사를 부릅니다.

In [ ]:
# 9단계: Skill 확인 → 없으면 설치(배포 폴더 복사) → 다시 확인
if is_placeholder(SKILL_NAME):
    show("Skill", None, "Skill 이름이 아직 정해지지 않았습니다.", "강사 안내를 기다리세요.")
else:
    target = SKILL_INSTALL_DIR / SKILL_NAME
    if not (target / "SKILL.md").exists():
        source = Path(SKILL_SOURCE)
        if not is_placeholder(SKILL_SOURCE) and (source / "SKILL.md").exists():
            print(f"설치 중: {source} → {target}")
            shutil.copytree(source, target, dirs_exist_ok=True)
            print("설치했습니다. Claude Code를 /exit 로 끄고 다시 claude 로 켜 주세요.")
        else:
            print(f"Skill 배포 위치를 찾지 못했습니다: {SKILL_SOURCE}")
    ok = (target / "SKILL.md").exists()
    show("Skill", ok, f"Skill 설치됨: {target}" if ok else "Skill이 설치되지 않았습니다.",
         "보조강사를 불러 주세요. [확인 필요: Skill 설치 방법]")

### 10단계. .env 설정 (Virtual Key 넣기)
**무엇을 하나요?** Claude Code가 키 자리만 비워 둔 `.env`(비밀값을 적어 두는 설정 파일)를 만듭니다. 키는 여러분이 **직접** 붙여 넣습니다.

**따라 하기**
1. Claude Code 입력창에 아래 내용을 붙여 넣습니다.

```
이 폴더에 .env 파일을 만들어줘. 내용은 아래 두 줄만 쓰고, 키 값은 비워 둬. 키는 내가 직접 넣을게.
[확인 필요: Gateway 주소 변수명]=[확인 필요: Gateway 주소]
[확인 필요: Virtual Key 변수명]=
```

2. 파일을 만들어도 되는지 묻는 창이 나오면, 내용을 확인하고 **Yes**를 고릅니다.
3. 터미널 오른쪽 위의 **+**로 새 터미널을 열고 아래를 입력합니다. 편집기에 `.env`가 열립니다.

```powershell
code $HOME\ax-training\.env
```

4. 메일로 받은 Virtual Key를 `=` 바로 뒤에 붙여 넣고 **Ctrl + S**로 저장합니다.
5. 이 노트북으로 돌아와 아래 셀을 실행합니다.

> 🔒 키는 **Claude Code 대화창에 붙여 넣지 마세요.** 대화 내용은 AI 모델로 전달되고 PC 기록에도 남습니다.
> 🔒 Claude Code는 8단계 설정 때문에 `.env`를 읽지 못합니다. 그래도 `.env`를 열어 보거나 출력하겠다고 물으면 **No**를 고릅니다.
> 🔒 실습 폴더와 `.env` 파일은 다른 사람과 공유하지 마세요.

**정상이면** `✅ Virtual Key 입력됨 (●●●● N자) / Gateway 주소: https://…` (키 내용은 화면에 나오지 않습니다)

**실패하면**
- `.env 파일이 없습니다` → 1번을 다시 합니다.
- `값이 비어 있습니다` → 4번에서 저장(Ctrl + S)했는지 확인합니다.

In [ ]:
# 10단계: .env 확인 (파일은 Claude Code가 만들고, 키는 사람이 직접 넣는다. 키 값은 출력하지 않는다)
env_path = PRACTICE_DIR / ".env"
if is_placeholder(ENV_KEY_NAME) or is_placeholder(ENV_URL_NAME):
    show(".env", None, ".env 변수 이름이 아직 정해지지 않았습니다.", "강사 안내를 기다리세요.")
elif not env_path.exists():
    show(".env", False, f".env 파일이 없습니다: {env_path}", "따라 하기 1번을 Claude Code에 다시 입력하세요.")
else:
    env = read_env(env_path)
    url, key = env.get(ENV_URL_NAME, ""), env.get(ENV_KEY_NAME, "")
    problems = []
    if not url:
        problems.append(f"{ENV_URL_NAME} 값이 비어 있습니다")
    if not key:
        problems.append(f"{ENV_KEY_NAME} 값이 비어 있습니다")
    elif key.startswith("["):
        problems.append("키 자리에 안내 문구가 그대로 있습니다")
    if problems:
        show(".env", False, " / ".join(problems), "VS Code에서 .env를 열어 값을 넣고 Ctrl+S로 저장한 뒤 다시 실행하세요.")
    else:
        show(".env", True, f"Virtual Key 입력됨 (●●●● {len(key)}자) / Gateway 주소: {url}")

### 11단계. Gateway 연결 확인
**무엇을 하나요?** 내 PC에서 사내 Gateway(AI 모델과 도구를 한 창구로 묶어 주는 사내 중계 서버) 주소까지 네트워크로 닿는지 확인합니다. 키가 맞는지는 12·13단계에서 확인합니다.

**정상이면** `✅ Gateway에 연결됨 (응답 코드 …)` (401, 404 같은 응답 코드도 "주소까지는 닿았다"는 뜻이라 정상입니다)

**실패하면** 주소 오타, 사내망 연결, 프록시 설정을 확인합니다. 인증서(SSL) 오류라고 나오면 보조강사를 부릅니다.

In [ ]:
# 11단계: Gateway 주소까지 네트워크가 닿는지 확인 (응답 코드와 상관없이 응답이 오면 정상)
import urllib.request, urllib.error
url = "" if is_placeholder(ENV_URL_NAME) else read_env(PRACTICE_DIR / ".env").get(ENV_URL_NAME, "")
if not url:
    show("Gateway 연결", None, "Gateway 주소가 없어 확인을 건너뜁니다.", "10단계를 먼저 ✅로 만드세요.")
else:
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            show("Gateway 연결", True, f"Gateway에 연결됨 (응답 코드 {r.status})")
    except urllib.error.HTTPError as e:
        show("Gateway 연결", True, f"Gateway에 연결됨 (응답 코드 {e.code}: 주소까지는 정상적으로 닿았습니다)")
    except urllib.error.URLError as e:
        reason = str(e.reason)
        hint = ("사내 인증서(SSL) 문제일 수 있습니다. 보조강사를 불러 주세요." if "CERTIFICATE" in reason.upper()
                else "주소 오타, 사내망 연결, 프록시 설정을 확인하세요. [확인 필요: 사내 프록시]")
        show("Gateway 연결", False, f"Gateway에 연결하지 못했습니다: {reason}", hint)
    except Exception as e:
        show("Gateway 연결", False, f"Gateway 확인 중 오류: {e}", "보조강사를 불러 주세요.")

### 12단계. Splunk 조회 확인 (Claude Code에서)
**무엇을 하나요?** Claude Code가 Skill과 Gateway를 거쳐 Splunk(보안 로그 저장·검색 시스템)를 조회할 수 있는지 확인합니다. **조회만** 합니다.

**따라 하기**
1. Claude Code 입력창에 붙여 넣습니다.

```
Splunk에서 최근 15분 동안 들어온 이벤트 건수만 알려줘. 조회만 하고 다른 작업은 하지 마.
```

2. 실행해도 되는지 묻는 창이 뜨면, 내용을 읽고 **1. Yes**(이번만 허용)를 고릅니다.
3. 건수(숫자)가 나오면 성공입니다. 아래 셀을 실행하고 `y`를 입력합니다.

**실패하면** Claude Code 화면의 오류를 그대로 두고 보조강사를 부릅니다. Skill이 쓰이지 않은 것 같으면 프롬프트 앞에 `[확인 필요: Skill 이름] Skill을 써서`를 붙여 다시 시도합니다.

In [ ]:
# 12단계: Splunk 조회 결과 확인 (직접 보고 y/n 입력)
answer = input("Claude Code가 Splunk 조회 결과(건수)를 보여 주었나요? (y/n): ").strip().lower()
show("Splunk 조회", answer.startswith("y"), "Splunk 조회 성공" if answer.startswith("y") else "Splunk 조회 실패",
     "Claude Code 화면의 오류를 그대로 두고 보조강사를 불러 주세요.")

### 13단계. 내 일정 조회 확인 (Claude Code에서)
**무엇을 하나요?** Claude Code가 사내 일정 MCP(AI가 외부 시스템을 쓰게 해 주는 연결 규격)로 **내 일정을 조회**할 수 있는지 확인합니다. 등록·수정·삭제는 하지 않습니다.

**따라 하기**
1. Claude Code 입력창에 붙여 넣습니다.

```
오늘 내 일정 목록만 조회해서 보여줘. 등록, 수정, 삭제는 하지 마.
```

2. 실행해도 되는지 묻는 창이 뜨면, 내용을 읽고 **1. Yes**(이번만 허용)를 고릅니다.
3. 오늘 일정 목록(일정이 없으면 "없음")이 나오면 성공입니다. 아래 셀을 실행하고 `y`를 입력합니다.

**실패하면** Claude Code 화면의 오류를 그대로 두고 보조강사를 부릅니다.

In [ ]:
# 13단계: 일정 조회 결과 확인 (직접 보고 y/n 입력)
answer = input("Claude Code가 오늘 내 일정(또는 '일정 없음')을 보여 주었나요? (y/n): ").strip().lower()
show("일정 조회", answer.startswith("y"), "일정 조회 성공" if answer.startswith("y") else "일정 조회 실패",
     "Claude Code 화면의 오류를 그대로 두고 보조강사를 불러 주세요.")

### 14단계. 최종 점검표
**무엇을 하나요?** 지금까지의 결과를 한눈에 모아 봅니다.

**정상이면** `🎉 셋팅 완료!`

**실패하면** ❌ 또는 ⬜(아직 실행 안 함) 항목의 셀로 돌아가 다시 실행합니다.

In [ ]:
# 14단계: 최종 점검표
print("=== 셋팅 결과 요약 ===")
count = {True: 0, False: 0, None: 0}
for step in STEP_ORDER:
    if step in RESULTS:
        ok, msg = RESULTS[step]
        count[ok] += 1
        print(f"{MARK[ok]} {step}: {msg}")
    else:
        print(f"⬜ {step}: 아직 실행하지 않음")
print()
if count[False] == 0 and all(s in RESULTS for s in STEP_ORDER):
    print("🎉 셋팅 완료!" + (" (⚠️ 항목은 강사 안내를 따르세요)" if count[None] else ""))
else:
    print("❌ 또는 ⬜ 항목의 셀을 다시 실행하세요. 같은 ❌가 반복되면 보조강사를 불러 주세요.")

---
# 📋 실습 프롬프트 모음 (참고용)

실습 시간에 **복사해서 Claude Code 입력창에 붙여 넣는** 프롬프트입니다. 이 아래 칸은 실행하는 셀이 아닙니다.
- `[ ]` 부분은 내 상황에 맞게 바꿉니다.
- **"아직 보내지 마", "먼저 보여줘", "받는 사람은 나", "[AX실습]"** 같은 안전 문구는 지우지 마세요.
- 실행해도 되는지 묻는 창이 뜨면 내용(받는 사람, 일정 제목)을 읽고 고릅니다. 발송·삭제에는 **"항상 허용"을 누르지 않습니다.**
- 내 말로 바꿔서 입력해도 괜찮습니다.

## 1교시 1-A. 말로 Splunk 묻기

**① 어떤 데이터가 있는지 묻기**

```
Splunk에서 내가 조회할 수 있는 보안 관련 데이터(인덱스)에는 무엇이 있는지 목록과 한 줄 설명으로 보여줘.
```

**② 질문하고 검색어 설명 듣기**

```
어제 하루 동안 보안 이벤트가 가장 많았던 출발지 IP 상위 5개를 표로 보여줘.
어떤 검색어(SPL)를 썼는지도 쉬운 말로 설명해줘.
```

## 1교시 1-B. 보안 이벤트 요약 브리핑 메일

**① 초안 만들기**

```
Splunk에서 어제 하루 보안 이벤트를 조회해서 유형별 건수와 눈에 띄는 점을 분석해줘.
결과는 아래 형식의 메일 초안으로 만들어줘.
- 제목: [일일 보안 브리핑] 날짜
- 3줄 요약
- 유형별 건수 상위 5개 표
- 권고 조치 3가지
보내는 사람과 받는 사람은 모두 나([내 메일 주소])야.
아직 보내지 말고 초안만 보여줘.
```

**② 말로 고치기 (예시)**

```
요약을 더 짧게 하고, 표를 요약 바로 아래로 옮겨줘. 아직 보내지 마.
```

**③ 확인 후 보내기 (받는 사람이 나인지 확인)**

```
좋아. 이 초안 그대로 나에게 메일로 보내줘.
```

## 1교시 1-C. (더 해보기) 같은 결과를 메신저로

**① 메신저용으로 줄이기**

```
방금 만든 브리핑을 메신저용으로 5줄 이내로 줄여줘.
받는 사람은 나([내 메신저 계정])야. 보내기 전에 내용 먼저 보여줘.
```

**② 확인 후 보내기**

```
좋아, 보내줘.
```

## 1교시 1-D. 점검 일정 등록과 정리

**① 실행 계획 받기**

```
어제 보안 이벤트가 많았던 서버(또는 IP) 상위 3개를 골라서 각각 점검할 항목을 정리해줘.
그리고 내일 오전에 서버별로 30분짜리 점검 일정을 내 일정에 등록하려고 해.
- 일정 제목은 반드시 '[AX실습]'으로 시작
- 일정 내용에는 점검 항목을 넣기
등록하기 전에 실행 계획(제목, 날짜·시간, 내용)을 표로 먼저 보여줘.
```

**② 확인 후 등록**

```
좋아, 계획대로 등록해줘.
```

**③ 등록 확인 (조회)**

```
내일 내 일정 중에서 제목이 '[AX실습]'으로 시작하는 일정만 조회해서 보여줘.
```

**④ 삭제할 목록 받기**

```
방금 조회한 '[AX실습]' 일정만 삭제하려고 해. 삭제할 목록을 먼저 보여줘.
```

**⑤ 확인 후 삭제**

```
확인했어. 삭제해줘.
```

## 2교시 2-A. 내 담당 영역 맞춤 리포트

**빈칸을 내 업무로 바꿔서 사용**

```
나는 [담당 업무: 예) VPN 운영] 담당이야.
Splunk에서 [기간: 예) 최근 7일] 동안의 [데이터: 예) VPN 접속 로그]를 조회해서 아래 형식으로 정리해줘.
- [보고 항목: 예) 일자별 접속 건수 추이, 지난주 대비 증감, 눈에 띄는 계정 상위 5개]
- 이름·사번 같은 개인정보는 가려줘
메일 초안으로 만들고, 보내는 사람과 받는 사람은 모두 나([내 메일 주소])야.
아직 보내지 말고 초안만 보여줘.
```

## 2교시 2-B. 의심 IP·계정 타임라인 조사

**빈칸을 바꿔서 사용**

```
[의심 IP 또는 계정: 예) 192.0.2.10]에 대해 Splunk에서 [기간: 예) 최근 3일] 동안 기록된 보안 로그를 모두 찾아서 시간순 타임라인으로 정리해줘.
- 시각 / 로그 출처 / 무슨 일이 있었는지 한 줄
- 마지막에 '확인된 사실'과 '추정'을 나눠서 의견을 적고, 추가로 확인할 점 3가지
조사 메모 형식의 메일 초안으로 만들어줘. 보내는 사람과 받는 사람은 모두 나([내 메일 주소])야.
아직 보내지 말고 초안만 보여줘.
```

## 2교시 2-C. 조치 항목 → 내 일정 → 할 일 알림

**① 조치 목록과 일정 계획 받기**

```
Splunk에서 [기간: 예) 최근 7일] 동안의 [조치 대상: 예) 보안 정책 위반 탐지]를 찾아서 조치가 필요한 항목 목록을 만들어줘.
항목마다 해야 할 조치와 마감일(오늘부터 2주 안)을 제안하고, 각 항목을 내 일정에 등록하려고 해.
- 일정 제목은 '[AX실습] 조치: 항목명' 형식
등록하기 전에 실행 계획을 표로 먼저 보여줘.
```

**② 확인 후 등록**

```
좋아, 계획대로 등록해줘.
```

**③ 할 일 목록 메신저 초안**

```
오늘부터 2주 안의 내 일정 중 '[AX실습] 조치:'로 시작하는 일정을 조회해서 할 일 목록으로 정리해줘.
그 목록을 메신저로 나([내 메신저 계정])에게 보내려고 해. 보내기 전에 내용 먼저 보여줘.
```

## 2교시 2-D. 내 프롬프트를 내일도 쓰게 저장

**① 프롬프트를 파일로 저장**

```
오늘 내가 [주제: 예) 2-A 맞춤 리포트]에 쓴 프롬프트를 다음에도 쓸 수 있게 정리해서
prompts 폴더에 '[파일 이름: 예) 주간_VPN_리포트].md' 파일로 저장해줘.
날짜·기간처럼 매번 바뀌는 부분은 [ ]로 비워 둬.
```

**② 저장한 프롬프트로 다시 실행**

```
prompts 폴더의 '[파일 이름].md' 내용대로 실행해줘. [ ] 부분은 [예) 기간=최근 3일]로 채워줘.
```

## 마무리. 실습 일정 정리 (필수)

**① 남은 실습 일정 조회**

```
오늘부터 30일 안의 내 일정 중 제목이 '[AX실습]'으로 시작하는 일정을 모두 조회해줘.
```

**② 삭제할 목록 받기**

```
조회한 '[AX실습]' 일정을 모두 삭제하려고 해. 삭제할 목록을 먼저 보여줘.
```

**③ 확인 후 삭제**

```
확인했어. 삭제해줘.
```